# PDFCleaner Algorithm Prototype

This notebook demonstrates the client-side scanned document cleaning pipeline developed for **PDFCleaner** using Python and OpenCV. This serves as the algorithm prototype before porting the pipeline to TypeScript and OpenCV.js WASM.

## Processing Steps:
1. **Grayscale Conversion**: Reduces color noise and computational complexity.
2. **Noise Reduction**: Uses Gaussian Blur or Median Blur to smooth small variations.
3. **Background Illumination Normalization**: Estimates the background using morphological close, then divides the image by it to remove uneven shadows.
4. **Gamma Correction**: Adjusts midtones to highlight text.
5. **Contrast Stretching**: Spreads pixel values across the entire 0-255 range.
6. **Adaptive Thresholding (Gaussian)**: Converts the image to black-and-white using local thresholds to handle local shadow variations.
7. **Morphological Post-processing**: Removes isolated black dots (noise) and fills small white gaps in text.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import time

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 10]
plt.rcParams['image.cmap'] = 'gray'

## 1. Load Sample Scans
We load the generated sample images simulating various scanned document issues.

In [ ]:
sample_dir = 'samples'
samples = {
    'Book Page (Shadows)': 'sample_book.png',
    'Receipt (Faded)': 'sample_receipt.png',
    'Journal (Yellow/Coffee)': 'sample_handwriting.png',
    'Photocopy (Heavy Noise)': 'sample_photocopy.png'
}

for name, filename in samples.items():
    path = os.path.join(sample_dir, filename)
    if os.path.exists(path):
        print(f"Loaded {name} ({path}): size {os.path.getsize(path)} bytes")
    else:
        print(f"Warning: {path} not found! Run generate_samples.py first.")

## 2. Core Pipeline Implementation
Let's implement each processing stage in a structured class so we can measure execution times and isolate parameters.

In [ ]:
class DocumentCleanerPipeline:
    def __init__(self, config):
        self.config = config
        self.timings = {}
        self.detected_skew_angle = 0.0
        
    def deskew(self, img):
        # We need grayscale to calculate skew angle
        if len(img.shape) == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = img.copy()
            
        # Invert the image (text becomes white, background black)
        thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        
        # Grab all coordinates of white pixels (text)
        coords = np.column_stack(np.where(thresh > 0))
        if len(coords) == 0:
            return img.copy(), 0.0
            
        # minAreaRect returns (center, size, angle)
        rect = cv2.minAreaRect(coords)
        angle = rect[-1]
        
        # Adjust angle to rotate back
        # In OpenCV 4.x minAreaRect returns angle in range [-90, 0] or [0, 90] depending on rotation
        if angle < -45:
            angle = -(90 + angle)
        elif angle > 45:
            angle = 90 - angle
        else:
            angle = -angle
            
        # Rotate the image
        (h, w) = img.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        # Warp with white background
        rotated = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))
        
        return rotated, angle

    def run(self, bgr_img):
        self.timings = {}
        stages = {}
        current_img = bgr_img.copy()
        
        # 0. Optional Auto-Deskew
        t0 = time.perf_counter()
        if self.config.get('enable_deskew', False):
            current_img, angle = self.deskew(current_img)
            self.detected_skew_angle = angle
        else:
            self.detected_skew_angle = 0.0
        self.timings['deskew'] = (time.perf_counter() - t0) * 1000
        stages['deskewed'] = current_img.copy()
        
        # 1. Grayscale
        t0 = time.perf_counter()
        if self.config.get('grayscale', True):
            gray = cv2.cvtColor(current_img, cv2.COLOR_BGR2GRAY)
        else:
            gray = current_img.copy()
        self.timings['grayscale'] = (time.perf_counter() - t0) * 1000
        stages['grayscale'] = gray.copy()
        
        # 2. Noise Reduction (Blur)
        t0 = time.perf_counter()
        if self.config.get('enable_noise_reduction', True):
            ksize = self.config.get('blur_kernel_size', 3)
            ksize = ksize if ksize % 2 == 1 else ksize + 1
            
            blur_type = self.config.get('blur_type', 'median')
            if blur_type == 'median':
                blurred = cv2.medianBlur(gray, ksize)
            elif blur_type == 'gaussian':
                blurred = cv2.GaussianBlur(gray, (ksize, ksize), 0)
            else:
                blurred = cv2.GaussianBlur(gray, (ksize, ksize), 0)
        else:
            blurred = gray.copy()
        self.timings['noise_reduction'] = (time.perf_counter() - t0) * 1000
        stages['noise_reduction'] = blurred.copy()
        
        # 3. Background Normalization
        t0 = time.perf_counter()
        if self.config.get('enable_background_norm', True):
            norm_ksize = self.config.get('norm_kernel_size', 25)
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (norm_ksize, norm_ksize))
            bg = cv2.morphologyEx(blurred, cv2.MORPH_CLOSE, kernel)
            normalized = cv2.divide(blurred, bg, scale=255)
        else:
            normalized = blurred.copy()
        self.timings['background_normalization'] = (time.perf_counter() - t0) * 1000
        stages['background_normalization'] = normalized.copy()
        
        # 4. Gamma Correction
        t0 = time.perf_counter()
        gamma = self.config.get('gamma', 1.0)
        if gamma != 1.0:
            invGamma = 1.0 / gamma
            table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
            gamma_corrected = cv2.LUT(normalized, table)
        else:
            gamma_corrected = normalized.copy()
        self.timings['gamma_correction'] = (time.perf_counter() - t0) * 1000
        stages['gamma_correction'] = gamma_corrected.copy()
        
        # 5. Contrast Enhancement (Stretching)
        t0 = time.perf_counter()
        contrast = self.config.get('contrast', 1.0)
        if contrast != 1.0 or True:
            contrast_stretched = cv2.normalize(gamma_corrected, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
        else:
            contrast_stretched = gamma_corrected.copy()
        self.timings['contrast_stretching'] = (time.perf_counter() - t0) * 1000
        stages['contrast_stretching'] = contrast_stretched.copy()
        
        # 6. Adaptive Thresholding
        t0 = time.perf_counter()
        if self.config.get('enable_thresholding', True):
            block = self.config.get('threshold_block_size', 21)
            block = block if block % 2 == 1 else block + 1
            c_val = self.config.get('threshold_c', 10)
            thresholded = cv2.adaptiveThreshold(
                contrast_stretched,
                255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY,
                block,
                c_val
            )
        else:
            thresholded = contrast_stretched.copy()
        self.timings['thresholding'] = (time.perf_counter() - t0) * 1000
        stages['thresholded'] = thresholded.copy()
        
        # 7. Morphology Cleanup (Noise Removal)
        t0 = time.perf_counter()
        if self.config.get('enable_morphology', True) and self.config.get('enable_thresholding', True):
            morph_size = self.config.get('morphology_kernel_size', 2)
            if morph_size > 0:
                kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (morph_size, morph_size))
                cleaned = cv2.morphologyEx(thresholded, cv2.MORPH_OPEN, kernel)
            else:
                cleaned = thresholded.copy()
        else:
            cleaned = thresholded.copy()
        self.timings['morphology'] = (time.perf_counter() - t0) * 1000
        stages['morphology'] = cleaned.copy()
        
        return stages



## 3. Define Mode Configurations
We create configurations corresponding to the 5 predefined processing modes outlined in the specification.

In [ ]:
MODE_CONFIGS = {
    'light-clean': {
        'grayscale': True,
        'enable_noise_reduction': True,
        'blur_type': 'gaussian',
        'blur_kernel_size': 3,
        'enable_background_norm': False,
        'gamma': 1.1,
        'contrast': 1.1,
        'enable_thresholding': False,
        'enable_morphology': False
    },
    'strong-background-removal': {
        'grayscale': True,
        'enable_noise_reduction': True,
        'blur_type': 'median',
        'blur_kernel_size': 3,
        'enable_background_norm': True,
        'norm_kernel_size': 25,
        'gamma': 0.8,
        'contrast': 1.4,
        'enable_thresholding': True,
        'threshold_block_size': 25,
        'threshold_c': 15,
        'enable_morphology': True,
        'morphology_kernel_size': 2
    },
    'text-contrast-boost': {
        'grayscale': True,
        'enable_noise_reduction': True,
        'blur_type': 'median',
        'blur_kernel_size': 3,
        'enable_background_norm': True,
        'norm_ksize': 21,
        'gamma': 0.6,
        'contrast': 1.6,
        'enable_thresholding': True,
        'threshold_block_size': 21,
        'threshold_c': 12,
        'enable_morphology': True,
        'morphology_kernel_size': 2
    },
    'print-optimized': {
        'grayscale': True,
        'enable_noise_reduction': True,
        'blur_type': 'median',
        'blur_kernel_size': 3,
        'enable_background_norm': True,
        'norm_kernel_size': 21,
        'gamma': 0.8,
        'contrast': 1.4,
        'enable_thresholding': True,
        'threshold_block_size': 21,
        'threshold_c': 15,
        'enable_morphology': True,
        'morphology_kernel_size': 2
    },
    'compressed-output': {
        'grayscale': True,
        'enable_noise_reduction': True,
        'blur_type': 'median',
        'blur_kernel_size': 3,
        'enable_background_norm': True,
        'norm_kernel_size': 15,
        'gamma': 0.9,
        'contrast': 1.3,
        'enable_thresholding': True,
        'threshold_block_size': 15,
        'threshold_c': 10,
        'enable_morphology': True,
        'morphology_kernel_size': 2
    }
}


## 4. Run Pipeline and Compare Before/After
Let's test each mode on its respective target sample scan to observe quality improvements.

In [ ]:
test_cases = [
    ('Book Page (Shadows)', 'sample_book.png', 'strong-background-removal'),
    ('Receipt (Faded)', 'sample_receipt.png', 'text-contrast-boost'),
    ('Journal (Yellow/Coffee)', 'sample_handwriting.png', 'light-clean'),
    ('Photocopy (Heavy Noise)', 'sample_photocopy.png', 'print-optimized')
]

for title, filename, mode in test_cases:
    img_path = os.path.join(sample_dir, filename)
    img = cv2.imread(img_path)
    
    config = MODE_CONFIGS[mode]
    cleaner = DocumentCleanerPipeline(config)
    stages = cleaner.run(img)
    
    # Output timings
    print(f"\n=== {title} processed with mode: {mode} ===")
    for stage, t in cleaner.timings.items():
        print(f"  Stage '{stage}': {t:.2f} ms")
    total_t = sum(cleaner.timings.values())
    print(f"  Total execution time: {total_t:.2f} ms")
    
    # Plot Before/After comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 8))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Original: {title}")
    axes[0].axis('off')
    
    cleaned_img = stages['morphology']
    axes[1].imshow(cleaned_img, cmap='gray')
    axes[1].set_title(f"Cleaned using mode: {mode}")
    axes[1].axis('off')
    
    # Save comparison to output
    output_dir = 'outputs'
    os.makedirs(output_dir, exist_ok=True)
    comparison_path = os.path.join(output_dir, f"compare_{filename}")
    fig.savefig(comparison_path, bbox_inches='tight')
    plt.show()

## 5. Visualizing Step-by-Step Stages
Let's trace a single document (e.g. the book page with heavy shadows) through all 7 stages of the pipeline to see how the background illumination shadow is removed.

In [ ]:
img_path = os.path.join(sample_dir, 'sample_book.png')
img = cv2.imread(img_path)

config = MODE_CONFIGS['strong-background-removal']
cleaner = DocumentCleanerPipeline(config)
stages = cleaner.run(img)

fig, axes = plt.subplots(3, 3, figsize=(16, 16))
axes = axes.ravel()

stages_list = [
    ('original', 'Original BGR'),
    ('grayscale', 'Grayscale'),
    ('noise-reduction', 'Noise Reduction (Blur)'),
    ('background_normalization', 'Background Normalized (Division)'),
    ('gamma_correction', 'Gamma Corrected'),
    ('contrast_stretching', 'Contrast Stretched'),
    ('thresholded', 'Adaptive Thresholded'),
    ('morphology', 'Morphological Cleaned (Final)')
]

for idx, (key, title) in enumerate(stages_list):
    stage_img = stages[key]
    if len(stage_img.shape) == 3:
        axes[idx].imshow(cv2.cvtColor(stage_img, cv2.COLOR_BGR2RGB))
    else:
        axes[idx].imshow(stage_img, cmap='gray')
    axes[idx].set_title(title)
    axes[idx].axis('off')

# Hide last empty subplot
axes[-1].axis('off')

fig.savefig('outputs/stages_visualization.png', bbox_inches='tight')
plt.show()

## 6. Key Learnings for Porting to OpenCV.js WASM

To replicate this pipeline client-side in OpenCV.js WASM:

1. **Grayscale conversion**:
   - In Python: `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)`
   - In OpenCV.js: `cv.cvtColor(src, dst, cv.COLOR_RGBA2GRAY)` (Web canvas handles RGBA, not BGR).
2. **Background Illumination Normalization**:
   - In Python: `cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)` followed by `cv2.divide(img, bg, scale=255)`
   - In OpenCV.js: 
     ```javascript
     let M = cv.Mat.ones(ksize, ksize, cv.CV_8U);
     cv.morphologyEx(src, bg, cv.MORPH_CLOSE, M);
     cv.divide(src, bg, dst, 255, -1);
     M.delete();
     ```
3. **Gamma Correction**:
   - OpenCV.js does not have a direct helper for Gamma correction, but we can compute the Lookup Table (LUT) in Javascript as a `cv.Mat` of size `(1, 256)` and use `cv.LUT`:
     ```javascript
     let lut = new cv.Mat(1, 256, cv.CV_8U);
     for (let i = 0; i < 256; i++) {
         lut.data[i] = Math.pow(i / 255.0, 1.0 / gamma) * 255.0;
     }
     cv.LUT(src, lut, dst);
     lut.delete();
     ```
4. **Contrast Stretching**:
   - In Python: `cv2.normalize(src, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)`
   - In OpenCV.js:
     ```javascript
     cv.normalize(src, dst, 0, 255, cv.NORM_MINMAX);
     ```
5. **Adaptive Thresholding**:
   - In Python: `cv2.adaptiveThreshold(src, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, blockSize, C)`
   - In OpenCV.js:
     ```javascript
     cv.adaptiveThreshold(src, dst, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C, cv.THRESH_BINARY, blockSize, C);
     ```
6. **Memory Management (CRITICAL)**:
   - Unlike Python where garbage collection automatically cleans up `numpy` arrays/mats, in WASM we **MUST** call `mat.delete()` manually for every temporary `cv.Mat` created in each stage, otherwise the WASM heap (default 256MB) will fill up and crash after processing a few pages!